## Scraping AWOIAF Character Names
Scrapes the full character list from the A Wiki of Ice and Fire and stores results in a CSV.

In [4]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

### Fetch the page

In [5]:
URL = 'https://awoiaf.westeros.org/index.php/List_of_characters'

# Full browser User-Agent required — the server returns 403 for generic bot strings
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}
page = requests.get(URL, headers=headers)

soup = BeautifulSoup(page.content, 'html.parser')
print(f'Status code: {page.status_code}')

Status code: 200


### Extract character names
The page is a MediaWiki article. Character names are listed as `<li>` items inside the main content div (`mw-parser-output`), each wrapped in an `<a>` tag.

In [6]:
# Scope to the main article body to avoid nav/footer links
content = soup.find('div', class_='mw-parser-output')

character_names = []

for li in content.find_all('li'):
    a = li.find('a')  # First <a> in each li is the character link
    if a and a.get('href', '').startswith('/index.php/'):  # Skip TOC anchor links (#A, #B ...)
        name = a.get_text(strip=True)
        if name and name not in character_names:
            character_names.append(name)

print(f'Found {len(character_names)} characters')
print(character_names[:10])  # Preview first 10

Found 3562 characters
['A certain man', 'Abelar Hightower', 'Abelon', 'Addam of Duskendale', 'Addam Frey', 'Addam Hightower', 'Addam Marbrand', 'Addam Osgrey', 'Addam Rivers', 'Addam Velaryon']


### Save to CSV

In [7]:
df = pd.DataFrame(character_names, columns=['name'])
df.to_csv('characters.csv', index=False)

print(f'Saved {len(df)} character names to characters.csv')
df.head(10)

Saved 3562 character names to characters.csv


,name
0,A certain man
1,Abelar Hightower
2,Abelon
3,Addam of Duskendale
4,Addam Frey
5,Addam Hightower
6,Addam Marbrand
7,Addam Osgrey
8,Addam Rivers
9,Addam Velaryon
